# 13장 · 순차 워크플로의 입력·출력 연결

순차 워크플로에서는 앞 단계의 반환값이 다음 호출의 입력이 됩니다. 중간값이 비면 이후 단계가 어떤 자료를 잃는지 살펴봅니다.

단계별 실제 모델 입력과 출력을 기록하세요. 분류 결과가 허용된 세 라벨에 속하는지 검사한 뒤 작성을 시작합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 요약 출력을 분류 입력으로 전달

단계별 실제 모델 입력과 출력을 기록하세요. 분류 결과가 허용된 세 라벨에 속하는지 검사한 뒤 작성을 시작합니다.

### 요약 단계의 정보 손실

<p>순차 흐름의 뒤 단계는 앞 단계가 넘긴 정보에 의존합니다. 원문에 “영수증을 찾을 수 없고 로그인 오류도 난다”는 내용이 있는데 요약이 영수증만 남기면 분류와 안내도 한쪽으로 치우칠 수 있습니다.</p>

<p>요약 길이를 늘리는 방법과 원문을 뒤 단계에도 전달하는 방법은 서로 다른 설계입니다. 어떤 정보를 남겨야 하는지 먼저 정하고, 입력 크기와 확인 부담을 함께 비교하세요.</p>


### 셀 01 · 실제 모델 호출

첫 번째 요청의 질문은 그대로 두고, 반환된 요약을 변수에 보관하세요.


In [ ]:
import os
from openai import OpenAI

client = OpenAI(timeout=45, max_retries=1)
model_name = os.getenv("OPENAI_MODEL") or "gpt-4o-mini"

from typing import Literal
from pydantic import BaseModel
class Routing(BaseModel):
    category: Literal["billing", "technical", "clarify"]

def generate_step(name, instruction, data):
    if name == "classify":
        reply = client.responses.parse(model=model_name, instructions=instruction, input=data, text_format=Routing)
        if reply.output_parsed is None:
            raise ValueError("모델이 분류 결과를 제공하지 않았습니다.")
        return reply.output_parsed.category
    return client.responses.create(model=model_name, instructions=instruction, input=data).output_text.strip()

request = "결제한 영수증을 어디에서 볼 수 있나요?"
summary = generate_step("summarize", "문의를 한 문장으로 요약하세요.", request)
print(summary)

결제한 영수증을 어디에서 확인할 수 있나요?


단계별 실제 모델 입력과 출력을 기록하세요. 분류 결과가 허용된 세 라벨에 속하는지 검사한 뒤 작성을 시작합니다.

### 순차 호출의 단계 분할

<p>여러 작업을 한 번에 요청하면 어디에서 잘못되었는지 구분하기 어려울 수 있습니다. 요약·분류·문장 작성을 따로 실행하면 중간 결과를 볼 수 있지만 호출 수와 대기 시간도 늘어납니다. 작업을 나눈다는 사실만으로 품질이 좋아진다고 단정하지 않습니다.</p>

<p>단계별 실제 모델 입력과 출력을 기록하세요. 분류 결과가 허용된 세 라벨에 속하는지 검사한 뒤 작성을 시작합니다.</p>

<table><caption>세 단계의 입출력</caption><thead><tr><th scope="col">단계</th><th scope="col">입력</th><th scope="col">출력</th></tr>
</thead><tbody><tr><td>summarize</td><td>원본 문의</td><td>요약 한 문장</td></tr>
<tr><td>classify</td><td>요약</td><td>billing/technical/clarify</td></tr>
<tr><td>write</td><td>요약과 분야를 담은 JSON 문자열</td><td>접수 안내</td></tr>
</tbody></table>


### 셀 02 · 분류 호출에 요약 전달

방금 얻은 summary를 세 번째 인자로 넣습니다.


In [ ]:
category = generate_step("classify", "billing, technical, clarify 중 하나를 쓰세요.", summary)
print(category)

billing


billing이 나옵니다. 다음 단계에 이 단어만 보내면 무엇 때문에 결제 문의로 분류했는지 알 수 없습니다.


#### 요약에서 빠진 정보는 어디에서 사라졌을까요?

<mark>두 번째 호출은 첫 번째 호출이 넘긴 요약을 입력으로 받습니다.</mark> 분류 결과가 뜻밖이면 분류 함수부터 바꾸기 전에 그 호출이 받은 문자열을 읽으세요. 원래 문의에 있던 영수증 정보가 요약에 남았는지부터 확인할 수 있습니다.

선수 개념 · 9장 프롬프트 설계와 버전 비교

#### 결제 문의의 요약이 다음 호출에 들어가는 경로

```mermaid
flowchart TD
a["원래 문의"] --> b["요약 호출"]
b -->|요약 문자열| c["분류 호출"]
c --> d["분류값 billing"]
```

두 번째 호출의 입력을 원래 문의와 대조하세요. 요약에서 빠진 정보는 분류 단계에 전달되지 않습니다.

### 모델 응답을 다음 단계로 전달하기

<p>단계별 실제 모델 입력과 출력을 기록하세요. 분류 결과가 허용된 세 라벨에 속하는지 검사한 뒤 작성을 시작합니다.</p>


## JSON으로 다중 입력 조립

요약은 무엇을 물었는지, 분류는 어느 분야인지 알려 줍니다. 하나의 문자열로 넘겨야 해도 칸 이름을 남겨 두면 모델과 사람이 같은 구조를 읽을 수 있습니다.


### 과제 03 · JSON으로 분류·요약 묶기

summary와 category를 같은 이름의 두 칸에 넣고 JSON으로 표시하세요. 한글은 읽을 수 있게 남깁니다.


In [3]:
import json
# 두 값을 JSON으로 묶으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>import json
writing_input = json.dumps({&quot;summary&quot;: summary, &quot;category&quot;: category}, ensure_ascii=False)
print(writing_input)</code></pre>
<p>실행 결과</p><pre><code>{&quot;summary&quot;: &quot;사용자가 결제 영수증 확인 방법을 문의했다.&quot;, &quot;category&quot;: &quot;billing&quot;}</code></pre>
<p>summary와 category가 구분되어 보입니다. 요약문에 들어 있는 영수증과 코드가 받은 분류값 billing을 한 번에 확인할 수 있습니다.</p>
</details>


### 작성 함수로 함께 전달되는 두 값

<p>분류 함수에는 앞서 만든 summary 하나가 들어갑니다. 작성 함수에는 summary와 category가 모두 필요해 두 값을 JSON 문자열로 묶습니다. 로그의 classify 입력과 write 입력에서 이 차이를 찾아보세요.</p>


### 셀 04 · 빈 문자열의 ValueError 처리

분류 전에 요약이 공백만 돌아온 상황입니다. 뒤의 호출에 보낼 값이 있는지 먼저 봅니다.


In [4]:
def checked_text(stage, output):
    if not isinstance(output, str) or not output.strip():
        raise ValueError("빈 단계 출력: " + stage)
    return output.strip()

try:
    checked_text("summarize", "   ")
except ValueError as error:
    print(error)

빈 단계 출력: summarize


오류 메시지에 summarize가 보입니다. 뒤에서 분류 실패가 난 것으로 오해하지 않고 처음 비어 버린 위치로 돌아갈 수 있습니다.


### 빈 중간값의 실패 처리

<p>셀 04의 입력은 요약 단계가 돌려준 공백 세 칸입니다. strip()을 적용하면 빈 문자열이 되어 checked_text가 ValueError를 냅니다. 요약 실패를 분류기의 문제로 기록하지 않도록 예외 메시지에 단계 이름을 넣습니다.</p>

<p>이 예제는 오류를 예외로 알립니다. 실패 시 앞 단계 로그를 외부에 반환하는 구조까지 구현하지 않았습니다. 실제 운영 로그를 만들 때는 성공 단계와 실패 단계를 함께 보관하되 민감한 입력 전체를 무조건 저장하지 않습니다.</p>

<table><caption>실패 조건</caption><thead><tr><th scope="col">발견 위치</th><th scope="col">처리</th></tr>
</thead><tbody><tr><td>시작 전 빈 문의</td><td>ValueError, 호출 없음</td></tr>
<tr><td>단계의 빈 출력</td><td>해당 단계에서 예외, 이후 중단</td></tr>
<tr><td>허용값 밖 분류</td><td>작성 단계 실행 전 중단</td></tr>
<tr><td>SDK 응답 미완료</td><td>호출 함수에서 예외</td></tr>
<tr><td>내용은 틀리지만 형식 정상</td><td>원문 대조로 별도 발견</td></tr>
</tbody></table>

## 단계별 호출 로그와 중간값 검증

다음 제공 함수는 각 단계의 입력, 출력, 시간을 log에 남깁니다. 앞 단계가 성공해야 다음 요청을 보냅니다. 실제 SDK를 쓸 때는 같은 자리에 responses.create로 본문을 읽는 함수를 연결할 수 있습니다.


### 셀 05 · 호출 로그의 단계별 input

처음에 입력한 질문과 두 번째 호출에 실제로 건넨 문자열을 구분하세요.

두 번째 호출이 받은 것은 첫 단계의 요약 ‘사용자가 결제 영수증 확인 방법을 문의했다.’입니다. 분류 결과 billing은 그다음 작성 단계에서 요약과 함께 쓰입니다. result["log"][1]["input"]에서 분류 전의 요약을 확인하세요.


In [ ]:
import json
from time import perf_counter
PROMPTS = {"summarize": "요약하세요.", "classify": "billing, technical, clarify 중 하나만 소문자로 반환하세요.", "write": "안내를 작성하세요."}

def run_workflow(call, request):
    if not request.strip():
        raise ValueError("빈 문의")
    log = []
    def step(name, data):
        started = perf_counter()
        output = call(name, PROMPTS[name], data)
        if not isinstance(output, str) or not output.strip():
            raise ValueError("빈 단계 출력: " + name)
        log.append({"stage": name, "input": data, "output": output,
                    "seconds": round(perf_counter() - started, 3)})
        return output.strip()
    summary = step("summarize", request)
    category = step("classify", summary)
    if category not in {"billing", "technical", "clarify"}:
        raise ValueError("허용하지 않은 분류")
    writing_input = json.dumps({"summary": summary, "category": category}, ensure_ascii=False)
    answer = step("write", writing_input)
    return {"summary": summary, "category": category, "answer": answer, "log": log}

result = run_workflow(generate_step, request)
print(result["log"][1]["input"])

결제한 영수증은 다음과 같은 방법으로 확인할 수 있습니다:

1. **이메일**: 결제 후 발송된 확인 이메일에서 영수증을 확인할 수 있습니다.
2. **앱 또는 웹사이트**: 사용한 서비스의 공식 앱이나 웹사이트에 로그인하여 '내 주문' 또는 '영수증' 섹션에서 확인할 수 있습니다.
3. **고객센터**: 필요한 경우 고객센터에 문의하여 영수증을 요청할 수 있습니다.

어떤 서비스인지에 따라 상황이 다를 수 있으니, 해당 서비스의 도움말을 참조하는 것도 좋습니다.


단계별 실제 모델 입력과 출력을 기록하세요. 분류 결과가 허용된 세 라벨에 속하는지 검사한 뒤 작성을 시작합니다.

### 단계 실행 함수

<p>정상 실행에서는 log[0]의 output이 log[1]의 input에 전달됩니다. 시간은 호출 전후의 차이를 소수 셋째 자리까지 반올림한 값이어서, 실행할 때마다 측정값이 달라집니다.</p>

<p>단계별 실제 모델 입력과 출력을 기록하세요. 분류 결과가 허용된 세 라벨에 속하는지 검사한 뒤 작성을 시작합니다.</p>

### 과제 06 · 작성 호출에 전달된 요약 검사

result의 실행 기록에서 작성 단계가 받은 JSON 문자열을 꺼내세요. 그 안의 summary가 첫 호출의 출력과 같은지 확인합니다. summary 자체를 빼먹은 기록은 False로 판정하세요.

In [ ]:
import json

def summary_arrived(log):
    # 작성 단계의 입력을 읽어 요약이 전달되었는지 확인하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>import json

def summary_arrived(log):
    write_input = json.loads(log[2][&quot;input&quot;])
    return write_input.get(&quot;summary&quot;) == log[0][&quot;output&quot;]

print(summary_arrived(result[&quot;log&quot;]))</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>세 번째 호출의 input은 문자열이어서 먼저 JSON을 해석합니다. 로그를 복사해 그 입력을 {&quot;category&quot;:&quot;billing&quot;}로 바꾸면 get이 None을 돌려주고 비교 결과는 False가 됩니다.</p>
</details>

## 요약에서 빠진 조건을 작성 요청에 되살리기

고객이 결제 금액을 공개하지 말아 달라고 덧붙였습니다. 실제 요약에서 금액과 날짜가 보존됐는지 확인하세요. 요약·분류는 그대로 두고 마지막 작성 단계에 들어가는 자료를 고쳐 봅니다.

### 셀 07 · 조건이 사라진 단계 찾기

기존 run_workflow에 원문을 보내고 작성 단계가 받은 JSON을 엽니다. 고객의 마지막 조건이 남았는지 확인하세요.

In [ ]:
original_ticket = "영수증은 어디에서 받나요? 답변에 결제 금액을 적지 마세요."
baseline = run_workflow(generate_step, original_ticket)
writer_packet = json.loads(baseline["log"][-1]["input"])
print(sorted(writer_packet))
print("결제 금액" in writer_packet["summary"])

['category', 'summary']
False


작성자가 받은 필드는 category와 summary이고, 실제 요약에 결제 금액이 남았는지 원문과 대조합니다. 이때 마지막 답변을 아무리 자세히 검사해도 작성자가 그 조건을 보았다는 보장은 없습니다.

### 과제 08 · 작성 단계 입력에 고객 원문 추가하기

run_workflow가 호출할 함수를 하나 감싸 보세요. write 단계에서만 JSON을 열어 original_request 필드에 고객 원문을 넣습니다. 작성 함수에 실제로 건넨 자료를 한 번 기록하고, 거기에 있는 고객 문장을 출력하세요.

In [ ]:
received_by_writer = []
def include_original(stage, instruction, data):
    pass  # write 단계에서 data를 보완하고 generate_step을 호출하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>received_by_writer = []
def include_original(stage, instruction, data):
    if stage == &quot;write&quot;:
        packet = json.loads(data)
        packet[&quot;original_request&quot;] = original_ticket
        data = json.dumps(packet, ensure_ascii=False)
        received_by_writer.append(data)
    return generate_step(stage, instruction, data)

run_workflow(include_original, original_ticket)
print(json.loads(received_by_writer[-1])[&quot;original_request&quot;])</code></pre>
<p>실행 결과</p><pre><code>영수증은 어디에서 받나요? 답변에 결제 금액을 적지 마세요.</code></pre>
<p>고객이 쓴 두 문장이 모두 작성 단계에 도착했습니다. run_workflow의 log는 함수를 호출하기 전 data를 기록하므로, 감싼 함수 안에서 바꾼 값은 received_by_writer에서 확인해야 합니다. 단계 입력과 실제 작성 결과를 함께 확인하세요.</p>
</details>

<mark>나중에 반드시 지킬 조건은 작성 단계가 실제로 받은 입력에서 확인하세요.</mark> 분류가 billing으로 맞았어도 요약 과정에서 고객의 요청이 사라질 수 있습니다. 마지막 함수가 받은 JSON을 꺼내 보아야 어느 연결을 고칠지 알 수 있습니다.

분류 경로를 추가할 때도 담당 함수가 고객의 원래 질문을 함께 받는지 점검하세요.

## 돌아보기

분류 대역이 payment를 돌려주면 run_workflow는 작성용 JSON을 만들기 전에 ValueError를 냅니다. 이 검사는 billing·technical·clarify라는 세 문자열만 허용한다는 규칙을 코드에서 보장합니다.